<img src="https://datascientest.fr/train/assets/logo_liora.png" style="height:150px">

<hr style="border-width:2px;border-color:#75DFC1">
<center><h1> AI Engineers — Démo Part 13 </h1></center>
<center><h2> De la démo au service : tests et CI </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

> **Part 13 · 20 min · ≈ 0,01 $ · Avant la séance : <code>make index</code> ET <code>make index-titled</code>** (les deux collections Qdrant)
> **Sans gateway, la démo se rabat sur les sorties enregistrées.**

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 1. Ce qu'on automatise </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Une démo **marche une fois**. Un service doit **continuer à marcher** après chaque modification.
- **Chaque changement** (prompt, modèle, découpage) peut **casser** une réponse sans erreur visible.

```
  git push ──▶ ┌────────────────────┐   ┌─────────────────────────┐   ┌──────────┐
               │ TESTS RAPIDES      │ ─▶│ TESTS RÉSEAU (à la      │ ─▶│ DÉPLOYER │
               │ sans modèle        │   │ demande) : la recherche │   │          │
               │ quelques secondes  │   │ ne doit pas régresser   │   │          │
               │ gratuits, à chaque │   │ gateway + index Qdrant  │   │          │
               │ push               │   │ ≈ 20 s, quelques cents  │   │          │
               └────────────────────┘   └─────────────────────────┘   └──────────┘
                  ❌ → on s'arrête         ❌ → on s'arrête
```

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 2. Deux familles de tests </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">



In [1]:
import os

# Move to the repository root when Jupyter starts the kernel in the demo folder.
if os.path.basename(os.getcwd()) == "demo":
    os.chdir("../..")

import subprocess
from pathlib import Path

print(subprocess.run(["grep", "-n", "^def test_\\|^@pytest.mark.network", "tests/test_regression.py"], capture_output=True, text=True).stdout)

42:def test_fiche_to_markdown_keeps_title_source_sections_and_links():
50:def test_normalize_ignores_case_apostrophes_and_spaces():
54:def test_evidence_in_context_requires_every_phrase():
62:def test_evidence_in_context_splits_table_cells_and_ignores_short_phrases():
68:def test_fiches_found_needs_all_gold_fiches():
74:def test_is_refusal():
79:def test_check_sources_returns_invented_sources_only():
85:def test_build_context_puts_the_fiche_id_before_each_passage():
89:def test_tokenize_keeps_article_numbers_in_one_word():
93:def test_expand_to_sections_replaces_chunks_and_removes_duplicates():
101:def test_expand_to_sections_keeps_the_chunk_when_the_section_is_too_long():
107:def test_summarize_counts_a_fake_run():
122:def test_reference_set_is_well_formed():
131:def test_gold_fiches_exist_in_the_corpus():
138:def test_answerable_evidence_is_in_the_corpus():
147:def test_chunks_with_title_start_with_their_section_path():
153:def test_mcp_server_exposes_the_two_tools():
180:@pytest.mar

- **Rapides** : on teste le **code** (fiche → Markdown, `evidence_in_context`, `check_sources`, `summarize`…) sur des cas **écrits à la main**.
- **Réseau** (`@pytest.mark.network`) : on teste la **recherche** sur le vrai index.
- **Aucun test ne compare le texte exact** d'une réponse du modèle : il change d'une exécution à l'autre.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 3. Les tests rapides : `make test` </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">



In [2]:
def run_pytest(*args):
    """Run pytest from the project root and return (exit code, last lines)."""
    done = subprocess.run(["uv", "run", "pytest", "-q", "--no-header", "--color=no", "-p", "no:cacheprovider", *args], capture_output=True, text=True)
    return done.returncode, "\n".join(done.stdout.strip().splitlines()[-4:])


code_, report = run_pytest("-m", "not network")
print(report)
print("\n✅ tests rapides au vert : on peut pousser" if code_ == 0 else "\n❌ un test rapide échoue")

.................                                                        [100%]
17 passed, 3 deselected in 5.23s

✅ tests rapides au vert : on peut pousser


- **Quelques secondes**, **aucune clé**, **aucun réseau** : ils tournent à **chaque** push.
- Ils attrapent les erreurs **bêtes** : une fonction cassée, un jeu de référence mal formé, une preuve absente du corpus.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 4. Le garde-fou : la mesure du chapitre 7 </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- La mesure « **la preuve est-elle dans le contexte ?** » devient un **seuil** dans un test.
- Valeurs mesurées pour `titled` : **15/22** (preuve) et **18/22** (bonne fiche).

In [3]:
import logging
import warnings

warnings.filterwarnings("ignore")  # no library warning on the projector

from src.evaluation import evidence_in_context, fiches_found, load_reference
from src.generation import build_context
from src.ingestion import load_sections
from src.retrieval import expand_to_sections, search

logging.getLogger("httpx").setLevel(logging.WARNING)
THRESHOLD = 15  # same value as MIN_EVIDENCE_IN_CONTEXT in tests/test_regression.py
questions = [q for q in load_reference() if q["type"] != "unanswerable"]
sections = load_sections()
RECORDED_COUNTS = {"titled (version actuelle)": 15, "baseline (ancienne version)": 11}  # chapter 7 reference values


def evidence_count(collection, expand):
    """Number of answerable questions whose evidence is in the passages given to the model."""
    count = 0
    for q in questions:
        docs = search(q["question"], k=5, collection=collection)
        if expand:
            docs = expand_to_sections(docs, sections)
        count += evidence_in_context(q, {"context": build_context(docs)})
    return count


LIVE = True
for name, collection, expand in [("titled (version actuelle)", "chunks_titled", True), ("baseline (ancienne version)", "chunks_500", False)]:
    try:
        count, note = evidence_count(collection, expand), ""
    except Exception as error:
        LIVE = False
        count, note = RECORDED_COUNTS[name], "  (sortie enregistrée : " + type(error).__name__ + ")"
    print(f"{name:30s} preuve dans le contexte : {count}/22  ", "✅ passe" if count >= THRESHOLD else f"❌ échoue (seuil {THRESHOLD})", note)
if LIVE:
    print("\n(valeurs de référence du chapitre 7 : titled 15, baseline 11)")

titled (version actuelle)      preuve dans le contexte : 17/22   ✅ passe 


baseline (ancienne version)    preuve dans le contexte : 11/22   ❌ échoue (seuil 15) 

(valeurs de référence du chapitre 7 : titled 15, baseline 11)


- Si quelqu'un **revient à la baseline** (ou casse le découpage) : **11 < 15**, le build **rouge**.
- Le seuil est **la valeur mesurée**, pas un souhait : on le **monte** quand on améliore, jamais on ne le baisse en silence.
- Le test réel est **le même calcul**, dans pytest. On le lance.

In [4]:
if LIVE:
    code_, report = run_pytest("-m", "network")
    print(report)
    print("\n✅ non-régression de la recherche : au vert" if code_ == 0 else "\n❌ la recherche a régressé")
else:
    print("⚠️ gateway inaccessible.")
    print("Sortie enregistrée : 3 passed, 17 deselected in 19.77s\n\n✅ non-régression de la recherche : au vert")

...                                                                      [100%]
3 passed, 17 deselected in 19.28s

✅ non-régression de la recherche : au vert


- Ces tests **appellent la gateway** (embeddings) et lisent **l'index** : on les lance **à la demande**, pas à chaque push.
- Pas de test sur la **formulation** d'une réponse : seulement ce qui est **mesurable** et **stable** (la recherche).

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 5. Le workflow GitHub Actions </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">



In [ ]:
KEYS = ("name:", "on:", "push:", "pull_request:", "workflow_dispatch:", "fast-tests:", "network-tests:", "if:", "needs:",
        "- run:", "LLM_BASE_URL:", "LLM_API_KEY:")
for line in Path(".github/workflows/ci.yml").read_text(encoding="utf-8").splitlines():
    if line.startswith(("name:", "on:", "jobs:")) or line.strip().startswith(KEYS):
        print(line)

- **Tests rapides** à chaque push et pull request : **aucun secret**.
- **Tests réseau** seulement sur déclenchement manuel (`workflow_dispatch`).
- Les clés viennent de **`${{ secrets.LLM_API_KEY }}`** : stockées dans GitHub, **jamais dans le dépôt**, masquées dans les journaux.
- `data/qdrant/` n'est pas versionné : le job **reconstruit l'index** (`make index-titled`).

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 6. Versionner ce qui change la réponse </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Une réponse dépend de **quatre choses**. Si l'une change sans trace, on ne sait plus **pourquoi** ça a bougé.

In [6]:
import hashlib

from src.generation import SYSTEM_PROMPT
from src.ingestion import load_chunks

fingerprint = {
    "prompt": hashlib.sha256(SYSTEM_PROMPT.encode()).hexdigest()[:8],
    "modèle": "gpt-4o-mini (temperature 0)",
    "embeddings": "text-embedding-3-small",
    "index": f"chunks_titled, {len(load_chunks(with_title=True))} chunks de 500 tokens max",
}
for key, value in fingerprint.items():
    print(f"{key:12s} {value}")

prompt       12dd9a07
modèle       gpt-4o-mini (temperature 0)
embeddings   text-embedding-3-small
index        chunks_titled, 569 chunks de 500 tokens max


| Élément | Où il est versionné |
|---|---|
| **Prompt** | dans le code (`SYSTEM_PROMPT`) : un `git diff` le montre |
| **Modèle** | nom **exact** dans le code, jamais « le dernier » |
| **Index** | collection **nommée** (`chunks_500`, `chunks_titled`) : on ne l'écrase pas |
| **Jeu de référence** | `data/eval/ref_v2.jsonl` : une nouvelle version = un nouveau fichier |

- **Une modification = un commit = une mesure** : le seuil dit si elle est **acceptée**.
- Relier une **trace** à la **version** qui l'a produite : on **étiquette** les traces (**chapitre 9, section 9.4**).

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 7. Après la mise en production : surveiller </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Les tests disent « **ça marchait hier** ». En production il faut voir **ce qui se passe maintenant**.

```
  CI (avant)                        PRODUCTION (après)
  tests rapides + seuil  ───▶  traces : latence · tokens · coût · refus
  « je peux déployer »         « tout va bien en ce moment »   → démo 12
```

- **Latence, tokens, coût, taux de refus**, par version : c'est la **démo 12** (LangSmith).
- Une **baisse de qualité** en production devient une **nouvelle question** du jeu de référence, donc un **nouveau test**.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > Récapitulatif </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

| Ce qu'on a vu | À retenir | Où le retrouver |
|---|---|---|
| **Tests rapides** (`make test`) | sans modèle ni réseau, à chaque push | `tests/test_regression.py` |
| **Tests réseau** (`make test-all`) | la recherche sur le vrai index, à la demande | `tests/test_regression.py` |
| **Seuil de non-régression** | la mesure du chapitre 7 : **15/22** de preuve dans le contexte, en dessous le build échoue | **Chapitre 7** (section 7.8) |
| **GitHub Actions** | `.github/workflows/ci.yml`, secrets via `secrets.*` | démo seule, pas de lab |
| **Versionner** | prompt, modèle, index, jeu de référence | **Chapitre 9** (section 9.4) |
| **Surveiller** | traces en production | **Démo 12**, chapitre 9 (section 9.8) |

- **Une modification n'est acceptée que si la mesure ne baisse pas.**
- **Fin du parcours : le projet, de bout en bout.**